# 1. What you'll learn

        - connect short random-tree paths with anomalousness
- choose a threshold from a normal-data quantile
- show clustered anomalies that isolation misses

        **The one question this notebook answers:** Can anomalies be found by how quickly random splits isolate them, without training on anomaly labels?

# 2. Setup

This lesson keeps isolation forest small enough to inspect end to end. Fixed seeds and one shared split make every comparison reproducible.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.ensemble import IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_curve,average_precision_score,confusion_matrix

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

Training contains only a two-dimensional normal cloud. Test data add sparse distant anomalies with labels retained solely for evaluation.

**Small example:** If five points need path lengths [8,7,9,2,3] to isolate, the last two are suspicious: random cuts separate them from the main cloud in very few questions.

In [ ]:
rng=np.random.default_rng(SEED);normal=rng.multivariate_normal([0,0],[[1,.55],[.55,1]],900);anomaly=rng.uniform([-6,-5],[6,5],size=(100,2));keep=np.linalg.norm(anomaly,axis=1)>3;anomaly=anomaly[keep][:75]
train=normal[:600];test=np.vstack([normal[600:],anomaly]);truth=np.r_[np.zeros(300,int),np.ones(len(anomaly),int)];print("Train normals:",train.shape,"test:",test.shape,"anomaly share:",round(truth.mean(),3))

# 4. Explore

The three views establish the target, the useful structure, and the specific data property that will govern the model's success or failure.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].scatter(test[:,0],test[:,1],c=truth,cmap="coolwarm",s=14);axes[0].set_title("Hidden evaluation labels")
axes[1].hist(np.linalg.norm(train,axis=1),bins=25,color="#176b66");axes[1].set_title("Normal radial distance")
axes[2].scatter(train[:,0],train[:,1],s=10,alpha=.4);axes[2].axvline(np.quantile(train[:,0],.1),ls="--");axes[2].axhline(np.quantile(train[:,1],.8),ls="--");axes[2].set_title("Random cuts isolate sparse points");plt.tight_layout();plt.show()

**Takeaway:** Planted anomalies occupy sparse outer regions.

**Takeaway:** Even valid normal points have a tail, so a threshold always creates false alarms.

**Takeaway:** Axis-aligned random cuts can isolate exterior points with few splits.

# 5. Prepare

We create the evaluation boundary before learning any transformation or model choice. Training-only operations remain inside the fitted workflow so held-out rows cannot influence them.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
print("Fit set contains labelled anomalies:",0);print("Threshold calibration set shape:",train.shape);contamination=.05

# 6. Train

        Training turns the assumptions behind isolation forest into an explicit fitted rule. The printed intermediate values expose what was learned rather than treating fit() as magic.

        Conceptually, fitting follows these steps:

        1. Subsample normal training rows for each tree.
2. Choose a random feature and split value recursively.
3. Record path length needed to isolate each point.
4. Average normalized path lengths and negate them into anomaly scores.

In [ ]:
model=IsolationForest(n_estimators=220,max_samples=256,contamination="auto",random_state=SEED,n_jobs=1).fit(train);train_score=-model.score_samples(train);score=-model.score_samples(test);threshold=np.quantile(train_score,1-contamination);pred=(score>=threshold).astype(int);print("Normal 95% threshold:",round(threshold,3),"flagged test:",pred.sum())

# 7. Evaluate

We compare against a deliberately simple baseline, report a task-appropriate metric, and save three diagnostic views that explain where the score came from.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
ap=average_precision_score(truth,score);cm=confusion_matrix(truth,pred);base=truth.mean();xx,yy=np.meshgrid(np.linspace(-6,6,150),np.linspace(-5,5,150));surface=-model.score_samples(np.c_[xx.ravel(),yy.ravel()]).reshape(xx.shape)
fig,axes=plt.subplots(1,3,figsize=(14,3.7));axes[0].contourf(xx,yy,surface,levels=20,cmap="mako");axes[0].scatter(test[:,0],test[:,1],c=truth,cmap="coolwarm",s=8);axes[0].set_title("Learned anomaly score")
axes[1].hist(score[truth==0],bins=25,alpha=.6,label="normal");axes[1].hist(score[truth==1],bins=25,alpha=.6,label="anomaly");axes[1].axvline(threshold,color="black",ls="--");axes[1].set_title("Quantile threshold");axes[1].legend()
sns.heatmap(cm,annot=True,fmt="d",cbar=False,ax=axes[2]);axes[2].set_title(f"AP {ap:.3f} vs prevalence {base:.3f}");plt.tight_layout();plt.savefig("key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** The model improves on the simple baseline where its assumptions match the data; the diagnostics show that the aggregate score alone is not enough.

# 8. Break it

We now violate the important condition directly: Isolation is geometric, so a dense malicious cluster near the normal center can look perfectly ordinary. The resulting number makes the cost of that violation visible.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
clustered=rng.multivariate_normal([0,0],[[.03,0],[0,.03]],75);test_bad=np.vstack([normal[600:],clustered]);truth_bad=np.r_[np.zeros(300,int),np.ones(75,int)];bad_score=-model.score_samples(test_bad)
print(f"Distant anomalies AP={ap:.3f}; dense in-distribution anomaly AP={average_precision_score(truth_bad,bad_score):.3f}; prevalence={truth_bad.mean():.3f}")

**Use this when…** anomalies are sparse and geometrically easier to isolate than normal observations.

        **Don't use this when…** fraud forms a dense cluster inside normal support or anomaly meaning is supervised and cost-sensitive.

        ## Try this

        1. Change the calibration quantile from 90% to 99%.
2. Add ten noise dimensions and measure AP.
3. Train with 5% anomalies and see threshold contamination.